In [3]:
import pandas as pd
import numpy as np
import math

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
df = pd.read_csv("mushrooms.csv")

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (8124, 23)
  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w       

In [5]:
def foil_gain(p0, n0, p1, n1):

    # If the new condition covers no positive examples
    if p1 == 0:
        return -float("inf")

    # If the new condition removes all negative examples
    if n1 == 0:
        return float("inf")

    # Probability before adding the condition
    before = p0 / (p0 + n0)

    # Probability after adding the condition
    after = p1 / (p1 + n1)

    gain = p1 * (
        math.log2(after) - math.log2(before)
    )

    return gain

In [7]:
def generate_candidates(data, used_features):

    candidates = []

    for feature in data.columns:

        if feature == "class":
            continue

        # Don't use the same feature twice
        if feature in used_features:
            continue

        for value in data[feature].unique():

            candidates.append(
                (feature, value)
            )

    return candidates

In [8]:
def learn_one_rule(pos, neg, data):

    rule = []

    current_pos = pos.copy()
    current_neg = neg.copy()

    used_features = set()

    while len(current_neg) > 0:

        p0 = len(current_pos)
        n0 = len(current_neg)

        candidates = generate_candidates(
            data,
            used_features
        )

        best_literal = None
        best_gain = -float("inf")

        best_pos = None
        best_neg = None

        # Check every candidate literal
        for literal in candidates:

            feature, value = literal

            new_pos = current_pos[
                current_pos[feature] == value
            ]

            new_neg = current_neg[
                current_neg[feature] == value
            ]

            p1 = len(new_pos)
            n1 = len(new_neg)

            gain = foil_gain(
                p0,
                n0,
                p1,
                n1
            )

            # Select literal with maximum gain
            if gain > best_gain:

                best_gain = gain
                best_literal = literal

                best_pos = new_pos
                best_neg = new_neg

        # No useful condition
        if best_literal is None:
            break

        # Add best literal to the rule
        rule.append(best_literal)

        feature, value = best_literal

        used_features.add(feature)

        current_pos = best_pos
        current_neg = best_neg

        print(
            "Added:",
            feature,
            "=",
            value,
            "| Gain:",
            round(best_gain, 4),
            "| Negative:",
            len(current_neg)
        )

        # Stop when no negative examples remain
        if len(current_neg) == 0:
            break

    return rule

In [9]:
def learn_one_rule(pos, neg, data):

    rule = []

    current_pos = pos.copy()
    current_neg = neg.copy()

    used_features = set()

    while len(current_neg) > 0:

        p0 = len(current_pos)
        n0 = len(current_neg)

        candidates = generate_candidates(
            data,
            used_features
        )

        best_literal = None
        best_gain = -float("inf")

        best_pos = None
        best_neg = None

        # Check every candidate literal
        for literal in candidates:

            feature, value = literal

            new_pos = current_pos[
                current_pos[feature] == value
            ]

            new_neg = current_neg[
                current_neg[feature] == value
            ]

            p1 = len(new_pos)
            n1 = len(new_neg)

            gain = foil_gain(
                p0,
                n0,
                p1,
                n1
            )

            # Select literal with maximum gain
            if gain > best_gain:

                best_gain = gain
                best_literal = literal

                best_pos = new_pos
                best_neg = new_neg

        # No useful condition
        if best_literal is None:
            break

        # Add best literal to the rule
        rule.append(best_literal)

        feature, value = best_literal

        used_features.add(feature)

        current_pos = best_pos
        current_neg = best_neg

        print(
            "Added:",
            feature,
            "=",
            value,
            "| Gain:",
            round(best_gain, 4),
            "| Negative:",
            len(current_neg)
        )

        # Stop when no negative examples remain
        if len(current_neg) == 0:
            break

    return rule

In [10]:
def foil_train(data):

    # Positive examples = poisonous
    pos = data[
        data["class"] == "p"
    ].copy()

    # Negative examples = edible
    neg = data[
        data["class"] == "e"
    ].copy()

    learned_rules = []

    print("Initial Positive Examples:", len(pos))
    print("Initial Negative Examples:", len(neg))

    # while Pos do
    while len(pos) > 0:

        print("\nLearning New Rule...")

        # Learn a new rule
        rule = learn_one_rule(
            pos,
            neg,
            data
        )

        # If no rule can be learned
        if len(rule) == 0:
            break

        # Add rule to learned rules
        learned_rules.append(rule)

        # Find positive examples covered by rule
        covered = pos.copy()

        for feature, value in rule:

            covered = covered[
                covered[feature] == value
            ]

        print("\nLearned Rule:")
        print_rule(rule)

        print(
            "Positive examples covered:",
            len(covered)
        )

        # Remove covered positive examples
        pos = pos.drop(
            covered.index
        )

        print(
            "Positive examples remaining:",
            len(pos)
        )

    return learned_rules

In [11]:
def print_rule(rule):

    conditions = []

    for feature, value in rule:

        conditions.append(
            f"{feature} = {value}"
        )

    print(
        "IF " +
        " AND ".join(conditions) +
        " THEN class = p"
    )

In [12]:
rules = foil_train(df)

Initial Positive Examples: 3916
Initial Negative Examples: 4208

Learning New Rule...
Added: cap-shape = c | Gain: inf | Negative: 0

Learned Rule:
IF cap-shape = c THEN class = p
Positive examples covered: 4
Positive examples remaining: 3912

Learning New Rule...
Added: cap-surface = g | Gain: inf | Negative: 0

Learned Rule:
IF cap-surface = g THEN class = p
Positive examples covered: 3
Positive examples remaining: 3909

Learning New Rule...
Added: odor = p | Gain: inf | Negative: 0

Learned Rule:
IF odor = p THEN class = p
Positive examples covered: 256
Positive examples remaining: 3653

Learning New Rule...
Added: odor = f | Gain: inf | Negative: 0

Learned Rule:
IF odor = f THEN class = p
Positive examples covered: 2160
Positive examples remaining: 1493

Learning New Rule...
Added: odor = c | Gain: inf | Negative: 0

Learned Rule:
IF odor = c THEN class = p
Positive examples covered: 192
Positive examples remaining: 1301

Learning New Rule...
Added: odor = y | Gain: inf | Negative

In [13]:
def predict_row(row, rules):

    # Check every learned rule
    for rule in rules:

        match = True

        for feature, value in rule:

            if row[feature] != value:
                match = False
                break

        # If a rule matches
        if match:
            return "p"

    # Default prediction
    return "e"

In [14]:
train_df, test_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["class"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training Data:", train_df.shape)
print("Testing Data :", test_df.shape)

Training Data: (5686, 23)
Testing Data : (2438, 23)


In [15]:
rules = foil_train(train_df)

Initial Positive Examples: 2741
Initial Negative Examples: 2945

Learning New Rule...
Added: cap-shape = c | Gain: inf | Negative: 0

Learned Rule:
IF cap-shape = c THEN class = p
Positive examples covered: 3
Positive examples remaining: 2738

Learning New Rule...
Added: cap-surface = g | Gain: inf | Negative: 0

Learned Rule:
IF cap-surface = g THEN class = p
Positive examples covered: 2
Positive examples remaining: 2736

Learning New Rule...
Added: odor = f | Gain: inf | Negative: 0

Learned Rule:
IF odor = f THEN class = p
Positive examples covered: 1529
Positive examples remaining: 1207

Learning New Rule...
Added: odor = s | Gain: inf | Negative: 0

Learned Rule:
IF odor = s THEN class = p
Positive examples covered: 389
Positive examples remaining: 818

Learning New Rule...
Added: odor = p | Gain: inf | Negative: 0

Learned Rule:
IF odor = p THEN class = p
Positive examples covered: 172
Positive examples remaining: 646

Learning New Rule...
Added: odor = m | Gain: inf | Negative: 

In [16]:
predictions = []

for _, row in test_df.iterrows():

    prediction = predict_row(
        row,
        rules
    )

    predictions.append(prediction)

In [17]:
y_true = test_df["class"]

accuracy = accuracy_score(
    y_true,
    predictions
)

print("\n==============================")
print("FOIL RESULTS")
print("==============================")

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)


FOIL RESULTS
Accuracy: 100.0 %


In [18]:
print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true,
        predictions
    )
)


Confusion Matrix:
[[1263    0]
 [   0 1175]]
